In [ ]:
# =====================================================================
# 1. SETUP ET IMPORTATIONS
# =====================================================================
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

# Installation et importation automatique de XGBoost si absent
try:
    from xgboost import XGBRegressor
except Exception:
    import sys
    !{sys.executable} -m pip install xgboost
    from xgboost import XGBRegressor

# Fixation de la graine aléatoire pour la reproductibilité
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


# =====================================================================
# 2. CHARGEMENT ET NETTOYAGE DES DONNÉES
# =====================================================================
print("--- Chargement des données ---")
# Recherche automatique des fichiers dans l'environnement Colab
pokemon_file = glob.glob("**/pokemon.csv", recursive=True)
combats_file = glob.glob("**/combats.csv", recursive=True)

if pokemon_file and combats_file:
    df_pokemon = pd.read_csv(pokemon_file[0])
    df_combats = pd.read_csv(combats_file[0])
    print(f"Fichiers trouvés et chargés : {pokemon_file[0]} et {combats_file[0]}")
else:
    print("Fichiers introuvables localement. Génération de données de simulation...")
    # Simulation de secours si les fichiers ne sont pas détectés au bon endroit
    df_pokemon = pd.DataFrame({
        '#': range(1, 101),
        'Name': [f'Pokemon_{i}' for i in range(1, 101)],
        'Type 1': np.random.choice(['Grass', 'Fire', 'Water', 'Bug', 'Normal'], 100),
        'Type 2': np.random.choice(['Poison', 'Flying', np.nan], 100),
        'HP': np.random.randint(30, 150, 100),
        'Attack': np.random.randint(30, 150, 100),
        'Defense': np.random.randint(30, 150, 100),
        'Sp. Atk': np.random.randint(30, 150, 100),
        'Sp. Def': np.random.randint(30, 150, 100),
        'Speed': np.random.randint(30, 150, 100),
        'Generation': np.random.randint(1, 7, 100),
        'Legendary': np.random.choice([True, False], 100)
    })
    df_pokemon.loc[61, 'Name'] = np.nan  # Simuler le cas du Pokémon manquant
    
    # Simulation de combats
    combats_data = {
        'First_pokemon': np.random.randint(1, 101, 1000),
        'Second_pokemon': np.random.randint(1, 101, 1000)
    }
    combats_data['Winner'] = [np.random.choice([f, s]) for f, s in zip(combats_data['First_pokemon'], combats_data['Second_pokemon'])]
    df_combats = pd.DataFrame(combats_data)

# --- Correction des valeurs manquantes ---
# 1. Compléter le nom manquant du Pokémon n° 62 (id 62 ou index correspondant)
df_pokemon.loc[df_pokemon['#'] == 62, 'Name'] = "Primeape"

# 2. Gérer les NaN pour la colonne Type 2
df_pokemon['Type 2'] = df_pokemon['Type 2'].fillna("Aucun")


# =====================================================================
# 3. CALCUL DU POURCENTAGE DE VICTOIRES (FEATURE ENGINEERING)
# =====================================================================
print("\n--- Calcul des statistiques de combat ---")
# Compter le nombre total de combats pour chaque Pokémon
total_first = df_combats['First_pokemon'].value_counts()
total_second = df_combats['Second_pokemon'].value_counts()
total_combats = total_first.add(total_second, fill_value=0)

# Compter le nombre de victoires
total_wins = df_combats['Winner'].value_counts()

# Calcul du taux de victoire (Win Percentage)
win_percentage = (total_wins / total_combats * 100).fillna(0)

# Conversion en DataFrame pour fusion
df_win_pct = pd.DataFrame({'#': win_percentage.index, 'Win_Percentage': win_percentage.values})

# Fusion avec le jeu de données principal des Pokémon
df_main = pd.merge(df_pokemon, df_win_pct, on='#', how='left')
# Les Pokémon n'ayant jamais combattu reçoivent un score de 0%
df_main['Win_Percentage'] = df_main['Win_Percentage'].fillna(0)


# =====================================================================
# 4. ANALYSE EXPLORATOIRE ET VISUALISATION
# =====================================================================
print("\n--- Génération des graphiques EDA ---")
# Sélection des statistiques numériques clés
stats_cols = ['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed', 'Win_Percentage']

# 1. Matrice de corrélation
plt.figure(figsize=(8, 6))
sns.heatmap(df_main[stats_cols].corr(), annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title("Matrice de Corrélation : Stats vs Pourcentage de Victoires")
plt.tight_layout()
plt.show()

# 2. PairGrid / Pairplot des statistiques clés en fonction du taux de victoire
sns.pairplot(df_main, x_vars=['HP', 'Attack', 'Defense', 'Speed'], y_vars=['Win_Percentage'], kind='reg', height=4)
plt.suptitle("Relations entre Statistiques et Pourcentage de Victoires", y=1.05)
plt.show()

# 3. Top 10 des meilleurs Pokémon
top_10 = df_main.sort_values(by='Win_Percentage', ascending=False).head(10)
print("\n=== TOP 10 DES POKÉMON AVEC LE MEILLEUR TAUX DE VICTOIRE ===")
print(top_10[['Name', 'Type 1', 'Speed', 'Attack', 'Win_Percentage']])


# =====================================================================
# 5. MACHINE LEARNING & PRÉDICTION (RÉGRESSION)
# =====================================================================
print("\n--- Préparation des modèles prédictifs ---")
# Définition des variables explicatives (Features) et de la cible (Target)
features = ['Type 1', 'Type 2', 'HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed', 'Legendary']
X = df_main[features]
y = df_main['Win_Percentage']

# Séparation des données (80% Train / 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

# Pipeline de Prétraitement
cat_cols = ['Type 1', 'Type 2', 'Legendary']
num_cols = ['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), cat_cols)
    ]
)

# Liste des modèles à évaluer
models = {
    'Régression Linéaire': LinearRegression(),
    'Forêt Aléatoire (Random Forest)': RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE),
    'XGBoost': XGBRegressor(n_estimators=100, learning_rate=0.08, max_depth=4, random_state=RANDOM_STATE)
}

mae_results = {}

# Entraînement et calcul des erreurs pour chaque modèle
for name, model in models.items():
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # Entraînement
    pipe.fit(X_train, y_train)
    
    # Prédiction sur l'ensemble de test
    y_pred = pipe.predict(X_test)
    
    # Calcul de la métrique d'évaluation (MAE)
    mae = mean_absolute_error(y_test, y_pred)
    mae_results[name] = mae
    print(f"{name} -> Erreur Absolue Moyenne (MAE): {mae:.4f}")

# =====================================================================
# 6. COMPARAISON ET SYNTHÈSE DES PERFORMANCES
# =====================================================================
df_perf = pd.DataFrame.from_dict(mae_results, orient='index', columns=['MAE (Erreur Moyenne)'])
df_perf = df_perf.sort_values(by='MAE (Erreur Moyenne)')

print("\n=== TABLEAU COMPARATIF DES MODÈLES (Classé du meilleur au moins bon) ===")
print(df_perf)

# Visualisation finale de la comparaison
plt.figure(figsize=(8, 4))
sns.barplot(x=df_perf.index, y=df_perf['MAE (Erreur Moyenne)'], palette='viridis')
plt.title("Comparaison des modèles par rapport à la MAE (Plus basse = Meilleure)")
plt.ylabel("Erreur Absolue Moyenne (Taux de victoire %)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()